# Image Detection (Colored Square) Notebook

This notebook generates a synthetic dataset of single colored squares (red/green/blue) and trains a tiny Keras model that predicts the bounding box (center x,y,width,height normalized) and the class.
It saves `model.h5` which the Flask app uses. If TensorFlow is not available, the notebook still includes a fallback color-threshold detector.


In [1]:
import os
from PIL import Image, ImageDraw
import numpy as np
import csv
os.listdir()

['.ipynb_checkpoints',
 '01-Tokenization.ipynb',
 '02-Stemming.ipynb',
 '03-Lemmatization.ipynb',
 '04-Stop-Words.ipynb',
 '50_Startups.csv',
 'Churn_Modelling.csv',
 'college_student_placement_dataset (1).csv',
 'Convolutional Neural network.ipynb',
 'Credit_Card_Applications.csv',
 'Data Exploration.ipynb',
 'dataset',
 'Decision Tree.ipynb',
 'Decision Trees.ipynb',
 'detection_notebook.ipynb',
 'DL.ipynb',
 'Forward selection attribute selection.ipynb',
 'health.csv',
 'Heart.csv',
 'KMEANs algo.ipynb',
 'KMeans copy.ipynb',
 'KNN Algorithm using imbalanced data handling.ipynb',
 'KNN Algorithm with cross validation.ipynb',
 'KNN Algorithm with hyper parameter tuning.ipynb',
 'KNN Algorithm.ipynb',
 'Multiple linear regression.ipynb',
 'NLP implementation using spacy.ipynb',
 'Plant_Disease_Model_Training.ipynb',
 'Polynomial regression.ipynb',
 'Practice.ipynb',
 'process - Copy.ipynb',
 'process.ipynb',
 'Project Practice - Copy.ipynb',
 'Project Practice.ipynb',
 'Random forest.

In [2]:
# Dataset path
dataset_dir = 'dataset'
print('Dataset folder contains:', len([f for f in os.listdir(dataset_dir) if f.endswith('.png')]), 'images')
print('Labels file:', os.path.join(dataset_dir, 'labels.csv'))

Dataset folder contains: 0 images
Labels file: dataset\labels.csv


In [4]:
# Model training (requires tensorflow)
try:
    import tensorflow as tf
    from tensorflow.keras import layers, models
    print('TensorFlow', tf.__version__)
    # Simple data loader
    import pandas as pd
    df = pd.read_csv(os.path.join(dataset_dir, 'labels.csv'))
    def load_image(row):
        im = Image.open(os.path.join(dataset_dir, row['file'])).resize((128,128)).convert('RGB')
        arr = np.array(im).astype('float32')/255.0
        ybox = np.array([row['xc'], row['yc'], row['bw'], row['bh']], dtype='float32')
        cls_map = {'red':0,'green':1,'blue':2}
        ycls = np.zeros(3, dtype='float32')
        ycls[cls_map[row['class']]] = 1.0
        return arr, np.concatenate([ybox, ycls])
    X = []
    Y = []
    for _,r in df.iterrows():
        x,y = load_image(r)
        X.append(x); Y.append(y)
    X = np.stack(X)
    Y = np.stack(Y)
    # Model: small CNN -> outputs 7 numbers (4 bbox + 3 class logits)
    inp = layers.Input(shape=(128,128,3))
    x = layers.Conv2D(16,3,activation='relu',padding='same')(inp)
    x = layers.MaxPool2D()(x)
    x = layers.Conv2D(32,3,activation='relu',padding='same')(x)
    x = layers.MaxPool2D()(x)
    x = layers.Conv2D(64,3,activation='relu',padding='same')(x)
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation='relu')(x)
    out_bbox = layers.Dense(4, activation='sigmoid', name='bbox')(x)
    out_cls = layers.Dense(3, activation='softmax', name='cls')(x)
    model = models.Model(inputs=inp, outputs=[out_bbox, out_cls])
    model.compile(optimizer='adam', loss={'bbox':'mse','cls':'categorical_crossentropy'})
    # Fit
    model.fit(X, [Y[:,:4], Y[:,4:]], epochs=8, batch_size=16)
    # Save a model that wraps outputs
    import tensorflow as tf
    out = tf.keras.layers.Concatenate()( [model.output[0], model.output[1]] )
    full = tf.keras.Model(inputs=model.input, outputs=out)
    full.save('model.h5')
    print('Saved model.h5')
except Exception as e:
    print('Training skipped or failed:', e)
    import traceback
    traceback.print_exc()
    print('Notebook contains fallback color-threshold detector as well.')

TensorFlow 2.19.0
Epoch 1/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 127ms/step - bbox_loss: 0.0841 - cls_loss: 2.2682 - loss: 2.3554
Epoch 2/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 123ms/step - bbox_loss: 0.0432 - cls_loss: 1.0184 - loss: 1.0617
Epoch 3/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 120ms/step - bbox_loss: 0.0353 - cls_loss: 0.6007 - loss: 0.6373
Epoch 4/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 121ms/step - bbox_loss: 0.0373 - cls_loss: 0.2140 - loss: 0.2516
Epoch 5/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 121ms/step - bbox_loss: 0.0370 - cls_loss: 0.0828 - loss: 0.1201
Epoch 6/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 124ms/step - bbox_loss: 0.0349 - cls_loss: 0.0089 - loss: 0.0438
Epoch 7/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 126ms/step - bbox_loss: 0.0357 - cls_loss: 0.0020 - loss: 0.0376
Epoch 8/8
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 123ms/step - bbox_loss: 0.0386 - cls_loss: 0.0014 - loss: 0.0400


Saved model.h5


## How to run
1. Start the Flask server: `python app.py`
2. Open `index.html` in a browser (served by Flask at http://localhost:5000/)

If `model.h5` exists, the app will use the trained Keras model; otherwise it will use a simple color-threshold detector.